In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
import os

In [2]:
load_dotenv()

True

In [3]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    groq_api_key=os.getenv("GROQ_API_KEY")
)

In [4]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState):

    prompt = f"""
Generate a short, funny joke about the topic: {state['topic']}

Return only the joke.
"""

    response = llm.invoke(prompt).content

    return {
        "joke": response
    }

In [6]:
def generate_explanation(state: JokeState):

    prompt = f"""
Explain the following joke in simple English:

{state['joke']}

Return only the explanation.
"""

    response = llm.invoke(prompt).content

    return {
        "explanation": response
    }

In [7]:
graph = StateGraph(JokeState)

# Add nodes
graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_explanation", generate_explanation)

# Add edges
graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "generate_explanation")
graph.add_edge("generate_explanation", END)

# Create memory checkpointer
checkpointer = InMemorySaver()

# Compile workflow
workflow = graph.compile(checkpointer=checkpointer)

In [9]:
config1 = {
    "configurable": {
        "thread_id": "1"
    }
}

result = workflow.invoke(
    {
        "topic": "pizza"
    },
    config=config1
)

print(result)

{'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? Because it was feeling a little crusty.', 'explanation': 'The joke is funny because "crusty" has two meanings. It can mean the outside part of a pizza, which is hard and crunchy. But it can also mean someone is being grumpy or irritable. So, the joke is saying the pizza is in a bad mood because it\'s feeling grumpy, and it\'s a play on words with "crusty" being a part of a pizza.'}


In [10]:
print("Joke:")
print(result["joke"])

print("\nExplanation:")
print(result["explanation"])

Joke:
Why was the pizza in a bad mood? Because it was feeling a little crusty.

Explanation:
The joke is funny because "crusty" has two meanings. It can mean the outside part of a pizza, which is hard and crunchy. But it can also mean someone is being grumpy or irritable. So, the joke is saying the pizza is in a bad mood because it's feeling grumpy, and it's a play on words with "crusty" being a part of a pizza.


In [11]:
state = workflow.get_state(config1)

print(state)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? Because it was feeling a little crusty.', 'explanation': 'The joke is funny because "crusty" has two meanings. It can mean the outside part of a pizza, which is hard and crunchy. But it can also mean someone is being grumpy or irritable. So, the joke is saying the pizza is in a bad mood because it\'s feeling grumpy, and it\'s a play on words with "crusty" being a part of a pizza.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18f34d-6b08-6bf4-8002-46040875e3c8'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-03T12:14:12.816178+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18f34d-64a2-6c6e-8001-da237389b271'}}, tasks=(), interrupts=())


In [19]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? Because it was feeling a little crusty.', 'explanation': 'The joke is funny because "crusty" has two meanings. It can mean the outside part of a pizza, which is hard and crunchy. But it can also mean someone is being grumpy or irritable. So, the joke is saying the pizza is in a bad mood because it\'s feeling grumpy, and it\'s a play on words with "crusty" being a part of a pizza.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18f34d-6b08-6bf4-8002-46040875e3c8'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-03T12:14:12.816178+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18f34d-64a2-6c6e-8001-da237389b271'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? Because it was feeling a little crusty.'}, next=('generate

In [14]:
config2 = {'configurable' : {'thread_id':'2'}}
workflow.invoke({'topic' : 'pasta'}, config= config2)

{'topic': 'pasta',
 'joke': 'Why was the pasta in a bad mood? Because it was feeling a little "drained".',
 'explanation': 'The joke is funny because "drained" has a double meaning. After you cook pasta, you drain the water from it. But "drained" can also mean feeling tired or unhappy. So, the joke is making a play on words to say the pasta is in a bad mood because it\'s feeling "drained", like the water was taken out of it, but also like it\'s feeling sad.'}

In [15]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why was the pasta in a bad mood? Because it was feeling a little "drained".', 'explanation': 'The joke is funny because "drained" has a double meaning. After you cook pasta, you drain the water from it. But "drained" can also mean feeling tired or unhappy. So, the joke is making a play on words to say the pasta is in a bad mood because it\'s feeling "drained", like the water was taken out of it, but also like it\'s feeling sad.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f18f362-0b4b-6462-8002-97ea9267ec17'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-03T12:23:26.491555+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f18f362-07b8-6158-8001-ffca0b2cdbcd'}}, tasks=(), interrupts=())

In [18]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why was the pasta in a bad mood? Because it was feeling a little "drained".', 'explanation': 'The joke is funny because "drained" has a double meaning. After you cook pasta, you drain the water from it. But "drained" can also mean feeling tired or unhappy. So, the joke is making a play on words to say the pasta is in a bad mood because it\'s feeling "drained", like the water was taken out of it, but also like it\'s feeling sad.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f18f362-0b4b-6462-8002-97ea9267ec17'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-03T12:23:26.491555+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f18f362-07b8-6158-8001-ffca0b2cdbcd'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pasta', 'joke': 'Why was the pasta in a bad mood? Because it was feeling a little "drained".

In [ ]:
#

#Time Travel

In [20]:
# Time Travel

workflow.get_state(
    {
        "configurable": {
            "thread_id": "1",
            "checkpoint_id": "1f18f34d-6187-624e-8000-15897694d1a5"
        }
    }
)

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f18f34d-6187-624e-8000-15897694d1a5'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-08-03T12:14:11.819374+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18f34d-617e-6ad7-bfff-b55aaf1b5358'}}, tasks=(PregelTask(id='25f003ad-b80b-de28-f346-6921733e2133', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why was the pizza in a bad mood? Because it was feeling a little crusty.'}),), interrupts=())

In [21]:
workflow.invoke(
    None,
    {
        "configurable": {
            "thread_id": "1",
            "checkpoint_id": "1f18f34d-6187-624e-8000-15897694d1a5"
        }
    }
)

{'topic': 'pizza',
 'joke': 'Why was the pizza in a bad mood? Because it was feeling a little crusty.',
 'explanation': 'The joke is funny because "crusty" has two meanings. It can mean the outside part of a pizza that is hard and crunchy. But it can also mean someone is being grumpy or irritable. So, the joke is saying the pizza is in a bad mood because it\'s "feeling crusty", which is a play on words.'}

In [22]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? Because it was feeling a little crusty.', 'explanation': 'The joke is funny because "crusty" has two meanings. It can mean the outside part of a pizza that is hard and crunchy. But it can also mean someone is being grumpy or irritable. So, the joke is saying the pizza is in a bad mood because it\'s "feeling crusty", which is a play on words.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18f3f3-08c2-6128-8003-536a3afabb5c'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-08-03T13:28:18.539754+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18f3f3-043c-6ee3-8002-d6574e7000dc'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? Because it was feeling a little crusty.'}, next=('generate_explanation',), config={'configurable'

#updatig state

# Time Travel



In [24]:
config = {
    "configurable": {
        "thread_id": "1",
        "checkpoint_id": "1f18f34d-6187-624e-8000-15897694d1a5"
    }
}

workflow.update_state(
    config,
    {
        "topic": "samosa"
    }
)

KeyError: 'checkpoint_ns'